In [1]:
!git clone https://github.com/SongX-1/NGAFIDDATASET.git

!(cd NGAFIDDATASET ; git checkout main; git reset --hard HEAD; git pull)
!(cd NGAFIDDATASET ; pip install -r requirements.txt -q)
! pip install mamba_ssm

Cloning into 'NGAFIDDATASET'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 39 (delta 16), reused 35 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 49.93 KiB | 12.48 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Already on 'main'
Your branch is up to date with 'origin/main'.
HEAD is now at e823293 Update dataset
Already up to date.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.0 MB/

In [2]:
import os
import sys
sys.path.append('/content/NGAFIDDATASET')

from ngafiddataset.dataset.dataset import NGAFID_Dataset_Downloader

name, saved_dir = NGAFID_Dataset_Downloader.download(
    name="2days",
    extract=True
)

print("Download finished.")
print("Dataset name:", name)
print("Dataset directory:", os.path.join(saved_dir, name))

/content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py:5: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm
Downloading...
From (original): https://drive.google.com/uc?id=1KIQKQOu9oMed_RMxtwn3Zpc2nC21viT_
From (redirected): https://drive.google.com/uc?id=1KIQKQOu9oMed_RMxtwn3Zpc2nC21viT_&confirm=t&uuid=470f36d5-2c8b-4902-bfb0-93dc6c351ada
To: /content/2days.tar.gz
100%|██████████| 1.13G/1.13G [00:06<00:00, 167MB/s]
2026-03-13 01:02:03.109 | INFO     | ngafiddataset.dataset.dataset:download:67 - Extracting File


Download finished.
Dataset name: 2days
Dataset directory: 2days


In [3]:
import math
import json
import random
import warnings
from dataclasses import dataclass, asdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
from compress_pickle import load
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler

try:
    from mamba_ssm import Mamba
except Exception as e:
    Mamba = None

warnings.filterwarnings("ignore")


In [ ]:
# =========================================================
# 0. Config
# =========================================================

@dataclass
class Config:
    # -------- Paths --------
    data_dir: str = "./2days"
    output_dir: str = "./outputs_v2"

    # -------- Data --------
    target_col: str = "before_after"
    fold_col: str = "fold"
    index_col: str = "Master Index"
    use_fold_column: bool = True
    num_workers: int = 4

    # Static-feature handling
    extra_drop_cols: Tuple[str, ...] = (
    "class", "target_class", "hclass", "label",
    "filename", "flight_id", "id", "tail", "tail_number",
    "date_diff",
    "number_flights_before",
    )
    use_static_features: bool = False


    force_seq_channels: int = 23

    # -------- Sequence norm --------
    use_stats_minmax: bool = True
    norm_eps: float = 1e-6

    # -------- Training --------
    seed: int = 42
    batch_size: int = 8
    epochs: int = 50
    lr: float = 1e-4
    weight_decay: float = 1e-4
    patience: int = 8
    amp: bool = False
    grad_clip: float = 0.5

    # -------- Bucketing --------
    bucket_boundaries: Tuple[int, ...] = (2000, 4000, 6000, 8000, 10000)
    shuffle_within_bucket: bool = True

    # -------- Model --------
    seq_in_dim_override: Optional[int] = None
    cnn_dim: int = 128
    stem_kernel: int = 7
    stem_stride: int = 2
    # 改
    mamba_layers: int = 2
    #
    mamba_d_state: int = 16
    mamba_d_conv: int = 4
    #
    mamba_expand: int = 2
    dropout: float = 0.15
    static_hidden: int = 128
    static_out: int = 64
    num_classes: int = 2


CFG = Config()


# =========================================================
# 1. Utils
# =========================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan


def downsample_mask_to_length(mask: torch.Tensor, target_len: int) -> torch.Tensor:

    current_len = mask.shape[1]
    if current_len == target_len:
        return mask
    if target_len <= 0:
        raise ValueError(f"target_len must be positive, got {target_len}")
    idx = torch.linspace(0, current_len - 1, steps=target_len, device=mask.device).long()
    return mask[:, idx]


# =========================================================
# 2. Data loading
# =========================================================

class NGAFIDRawLoader:
    """
    Uses the same 2days file structure as dataset.py:
      - flight_data.pkl
      - flight_header.csv
      - stats.csv
    but DOES NOT truncate to 4096.
    """

    def __init__(self, data_dir: str, index_col: str = "Master Index", force_seq_channels: int = 23):
        self.data_dir = data_dir
        self.index_col = index_col
        self.force_seq_channels = force_seq_channels

        self.flight_data_path = os.path.join(data_dir, "flight_data.pkl")
        self.header_path = os.path.join(data_dir, "flight_header.csv")
        self.stats_path = os.path.join(data_dir, "stats.csv")

        self.flight_data = load(self.flight_data_path)
        self.header_df = pd.read_csv(self.header_path)
        if self.index_col in self.header_df.columns:
            self.header_df = self.header_df.set_index(self.index_col)
        self.stats_df = pd.read_csv(self.stats_path)

        self.maxs = self.stats_df.iloc[0, 1:1 + self.force_seq_channels].to_numpy(dtype=np.float32)
        self.mins = self.stats_df.iloc[1, 1:1 + self.force_seq_channels].to_numpy(dtype=np.float32)

    def get_common_ids(self) -> List[int]:
        return [idx for idx in self.header_df.index if idx in self.flight_data]

    def load(self, target_col: str = "before_after"):
        common_ids = self.get_common_ids()

        seqs = []
        valid_ids = []
        bad_ids = []

        for idx in common_ids:
            arr = np.asarray(self.flight_data[idx], dtype=np.float32)

            if arr.ndim != 2:
                bad_ids.append(idx)
                continue

            if arr.shape[1] < self.force_seq_channels:
                bad_ids.append(idx)
                continue

            arr = arr[:, :self.force_seq_channels].copy()

            seqs.append(arr)
            valid_ids.append(idx)

        header = self.header_df.loc[valid_ids].copy()

        labels_series = header[target_col]
        if labels_series.dtype == object:
            uniq = sorted(pd.Series(labels_series).dropna().unique().tolist())
            if len(uniq) != 2:
                raise ValueError(f"{target_col} is not binary: {uniq}")
            mapper = {uniq[0]: 0, uniq[1]: 1}
            labels = labels_series.map(mapper).values.astype(np.int64)
        else:
            labels = labels_series.astype(np.int64).values

        lengths = np.array([len(x) for x in seqs], dtype=np.int64)

        return {
            "ids": valid_ids,
            "seqs": seqs,
            "header_df": header,
            "labels": labels,
            "lengths": lengths,
            "mins": self.mins,
            "maxs": self.maxs,
            "bad_ids": bad_ids,
        }


def infer_static_columns(header_df: pd.DataFrame,
                         target_col: str,
                         fold_col: str,
                         extra_drop_cols: Tuple[str, ...]) -> List[str]:
    cols = []
    excluded = {target_col, fold_col, *extra_drop_cols}
    for c in header_df.columns:
        cl = c.lower()
        if c in excluded:
            continue
        if cl in {"master index", "master_index", "index", "idx", "date", "time"}:
            continue
        if pd.api.types.is_numeric_dtype(header_df[c]):
            cols.append(c)
    return cols


class FoldPreprocessor:
    def __init__(self, mins: np.ndarray, maxs: np.ndarray, use_stats_minmax: bool = True, eps: float = 1e-6):
        self.mins = mins.astype(np.float32)
        self.maxs = maxs.astype(np.float32)
        self.use_stats_minmax = use_stats_minmax
        self.eps = eps
        self.static_imputer = SimpleImputer(strategy="median")
        self.static_scaler = StandardScaler()
        self.seq_mean = None
        self.seq_std = None

    def fit_sequence(self, seqs: List[np.ndarray]):
        if self.use_stats_minmax:
            return
        concat = np.concatenate(
            [np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0) for x in seqs],
            axis=0
        )
        self.seq_mean = concat.mean(axis=0).astype(np.float32)
        self.seq_std = concat.std(axis=0).astype(np.float32)
        self.seq_std[self.seq_std < self.eps] = 1.0

    def transform_sequence(self, seq: np.ndarray) -> np.ndarray:
        seq = np.asarray(seq, dtype=np.float32).copy()
        seq = np.nan_to_num(seq, nan=0.0, posinf=0.0, neginf=0.0)

        if self.use_stats_minmax:
            denom = np.maximum(self.maxs - self.mins, self.eps)
            seq = (seq - self.mins) / denom
        else:
            seq = (seq - self.seq_mean) / self.seq_std

        return seq.astype(np.float32)

    def fit_static(self, static_df: pd.DataFrame):
        if static_df.shape[1] == 0:
            return
        x = self.static_imputer.fit_transform(static_df.values)
        self.static_scaler.fit(x)

    def transform_static(self, static_df: pd.DataFrame) -> np.ndarray:
        if static_df.shape[1] == 0:
            return np.zeros((len(static_df), 0), dtype=np.float32)
        x = self.static_imputer.transform(static_df.values)
        x = self.static_scaler.transform(x)
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        return x.astype(np.float32)

    def fit(self, train_seqs: List[np.ndarray], train_static_df: pd.DataFrame):
        self.fit_sequence(train_seqs)
        self.fit_static(train_static_df)


# =========================================================
# 3. Dataset / Collate / Bucket sampler
# =========================================================

class NGAFIDSequenceDataset(Dataset):
    def __init__(self,
                 ids: List[int],
                 seqs: List[np.ndarray],
                 static_feats: np.ndarray,
                 labels: np.ndarray):
        self.ids = ids
        self.seqs = seqs
        self.static_feats = static_feats
        self.labels = labels.astype(np.int64)
        self.lengths = np.array([x.shape[0] for x in seqs], dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx: int):
        return {
            "id": self.ids[idx],
            "seq": self.seqs[idx],
            "static": self.static_feats[idx],
            "label": self.labels[idx],
            "length": self.lengths[idx],
        }


def collate_fn(batch):
    max_len = max(item["seq"].shape[0] for item in batch)
    in_dim = batch[0]["seq"].shape[1]
    bsz = len(batch)

    for item in batch:
        if item["seq"].shape[1] != in_dim:
            raise ValueError(f"Inconsistent seq dims in batch: got {item['seq'].shape[1]} vs {in_dim}")

    x = np.zeros((bsz, max_len, in_dim), dtype=np.float32)
    mask = np.zeros((bsz, max_len), dtype=np.bool_)
    static = np.zeros((bsz, len(batch[0]["static"])), dtype=np.float32)
    y = np.zeros((bsz,), dtype=np.int64)
    ids = []
    lengths = np.zeros((bsz,), dtype=np.int64)

    for i, item in enumerate(batch):
        t = item["seq"].shape[0]
        x[i, :t] = item["seq"]
        mask[i, :t] = True
        static[i] = item["static"]
        y[i] = item["label"]
        ids.append(item["id"])
        lengths[i] = item["length"]

    return {
        "ids": ids,
        "x": torch.from_numpy(x),
        "mask": torch.from_numpy(mask),
        "static": torch.from_numpy(static),
        "y": torch.from_numpy(y),
        "lengths": torch.from_numpy(lengths),
    }


class LengthBucketBatchSampler(Sampler[List[int]]):
    def __init__(self,
                 lengths: np.ndarray,
                 batch_size: int,
                 boundaries: Tuple[int, ...],
                 shuffle: bool = True,
                 drop_last: bool = False):
        self.lengths = np.asarray(lengths)
        self.batch_size = batch_size
        self.boundaries = list(boundaries)
        self.shuffle = shuffle
        self.drop_last = drop_last

        bucket_ids = np.digitize(self.lengths, bins=self.boundaries, right=True)
        self.buckets = {}
        for idx, b in enumerate(bucket_ids):
            self.buckets.setdefault(int(b), []).append(idx)

    def __iter__(self):
        all_batches = []
        bucket_keys = list(self.buckets.keys())
        if self.shuffle:
            random.shuffle(bucket_keys)

        for b in bucket_keys:
            indices = self.buckets[b][:]
            if self.shuffle:
                random.shuffle(indices)
            for i in range(0, len(indices), self.batch_size):
                batch = indices[i:i + self.batch_size]
                if len(batch) < self.batch_size and self.drop_last:
                    continue
                all_batches.append(batch)

        if self.shuffle:
            random.shuffle(all_batches)

        for batch in all_batches:
            yield batch

    def __len__(self):
        total = 0
        for indices in self.buckets.values():
            if self.drop_last:
                total += len(indices) // self.batch_size
            else:
                total += math.ceil(len(indices) / self.batch_size)
        return total


# =========================================================
# 4. Model
# =========================================================

class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, groups=1):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=padding, groups=groups, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class MultiScaleResBlock(nn.Module):
    def __init__(self, channels: int, dropout: float = 0.1):
        super().__init__()
        branch = channels // 4
        self.b1 = ConvBNAct(channels, branch, 3)
        self.b2 = ConvBNAct(channels, branch, 7)
        self.b3 = ConvBNAct(channels, branch, 15)
        self.pool = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            ConvBNAct(channels, branch, 1),
        )
        self.proj = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
        )
        self.out = nn.Sequential(
            nn.Conv1d(branch * 4, channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        res = self.proj(x)
        y = torch.cat([self.b1(x), self.b2(x), self.b3(x), self.pool(x)], dim=1)
        y = self.out(y)
        return F.gelu(y + res)


class BiMambaBlock(nn.Module):
    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int, dropout: float):
        super().__init__()
        if Mamba is None:
            raise ImportError("mamba_ssm is not installed. Install with: pip install mamba-ssm")
        self.fwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.bwd = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.ffn = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
        )

    def forward(self, x, mask):
        # x: (B, T, D), mask: (B, T) bool
        m = mask.unsqueeze(-1).to(x.dtype)

        y1 = self.fwd(x)
        xr = torch.flip(x, dims=[1])
        y2 = self.bwd(xr)
        y2 = torch.flip(y2, dims=[1])

        y = 0.5 * (y1 + y2)
        y = y * m
        x = x + self.dropout(y)
        x = self.norm(x)
        x = x + self.ffn(x)
        x = x * m
        return x


class StaticMLP(nn.Module):
    def __init__(self, in_dim: int, hidden: int, out_dim: int, dropout: float):
        super().__init__()
        if in_dim == 0:
            self.net = None
            self.out_dim = 0
        else:
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden, out_dim),
                nn.GELU(),
            )
            self.out_dim = out_dim

    def forward(self, x):
        if self.net is None:
            return x.new_zeros((x.shape[0], 0))
        return self.net(x)


class CNNBiMambaClassifier(nn.Module):
    def __init__(self,
                 seq_in_dim: int,
                 static_dim: int,
                 cnn_dim: int,
                 stem_kernel: int,
                 stem_stride: int,
                 mamba_layers: int,
                 mamba_d_state: int,
                 mamba_d_conv: int,
                 mamba_expand: int,
                 static_hidden: int,
                 static_out: int,
                 num_classes: int,
                 dropout: float):
        super().__init__()
        self.stem = nn.Sequential(
            ConvBNAct(seq_in_dim, cnn_dim, stem_kernel, stride=stem_stride),
            ConvBNAct(cnn_dim, cnn_dim, 3, stride=1),
        )
        self.ms1 = MultiScaleResBlock(cnn_dim, dropout=dropout)
        self.down = ConvBNAct(cnn_dim, cnn_dim, 3, stride=2)
        self.ms2 = MultiScaleResBlock(cnn_dim, dropout=dropout)

        self.mamba_blocks = nn.ModuleList([
            BiMambaBlock(cnn_dim, mamba_d_state, mamba_d_conv, mamba_expand, dropout)
            for _ in range(mamba_layers)
        ])

        self.static_branch = StaticMLP(static_dim, static_hidden, static_out, dropout)
        fusion_dim = cnn_dim * 2 + self.static_branch.out_dim
        self.head = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x, mask, static):
        # x: (B, T, C), mask: (B, T)
        x = x.transpose(1, 2)  # (B, C, T)

        x = self.stem(x)
        mask = downsample_mask_to_length(mask, x.shape[-1])

        x = self.ms1(x)
        x = self.down(x)
        mask = downsample_mask_to_length(mask, x.shape[-1])

        x = self.ms2(x)
        x = x.transpose(1, 2)  # (B, T', D)

        for block in self.mamba_blocks:
            x = block(x, mask)

        m = mask.unsqueeze(-1).to(x.dtype)
        denom = m.sum(dim=1).clamp_min(1.0)
        mean_pool = (x * m).sum(dim=1) / denom

        neg_inf = torch.full_like(x, -1e9)
        max_pool = torch.where(mask.unsqueeze(-1), x, neg_inf).amax(dim=1)
        max_pool = torch.where(torch.isfinite(max_pool), max_pool, torch.zeros_like(max_pool))

        static_feat = self.static_branch(static)
        fused = torch.cat([mean_pool, max_pool, static_feat], dim=1)
        logits = self.head(fused)
        return logits


# =========================================================
# 5. Train / Eval
# =========================================================

class EarlyStopping:
    def __init__(self, patience: int = 8, mode: str = "max"):
        self.patience = patience
        self.mode = mode
        self.best = None
        self.num_bad = 0
        self.stop = False

    def step(self, score: float) -> bool:
        improved = False
        if self.best is None:
            improved = True
        elif self.mode == "max" and score > self.best:
            improved = True
        elif self.mode == "min" and score < self.best:
            improved = True

        if improved:
            self.best = score
            self.num_bad = 0
            return True

        self.num_bad += 1
        if self.num_bad >= self.patience:
            self.stop = True
        return False


def run_epoch(model, loader, optimizer, criterion, device, scaler=None, train=True, amp=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_y = []
    all_pred = []
    rows = []

    use_amp = amp and torch.device(device).type == "cuda"

    for batch in loader:
        x = batch["x"].to(device, non_blocking=True)
        mask = batch["mask"].to(device, non_blocking=True)
        static = batch["static"].to(device, non_blocking=True)
        y = batch["y"].to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            if use_amp:
                with torch.cuda.amp.autocast(enabled=True):
                    logits = model(x, mask, static)
                    loss = criterion(logits, y)
            else:
                logits = model(x, mask, static)
                loss = criterion(logits, y)

            if train:
                if use_amp and scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
                    optimizer.step()

        total_loss += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)

        all_y.extend(y.detach().cpu().numpy().tolist())
        all_pred.extend(pred.detach().cpu().numpy().tolist())

        if not train:
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            for sid, yy, pp, prob in zip(
                batch["ids"],
                y.detach().cpu().numpy(),
                pred.detach().cpu().numpy(),
                probs
            ):
                rows.append({
                    "id": sid,
                    "y_true": int(yy),
                    "y_pred": int(pp),
                    "prob_after": float(prob),
                })

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_y, all_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_y, all_pred, average="binary", zero_division=0
    )

    return {
        "loss": avg_loss,
        "acc": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "pred_rows": rows,
    }


# =========================================================
# 6. Fold split
# =========================================================

def get_fold_splits(header_df: pd.DataFrame, fold_col: str = "fold"):
    if fold_col not in header_df.columns:
        raise ValueError(f"{fold_col} not found in header_df")
    folds = sorted(pd.Series(header_df[fold_col]).dropna().astype(int).unique().tolist())
    if len(folds) != 5:
        raise ValueError(f"Expected 5 folds in column {fold_col}, got {folds}")
    return folds


# =========================================================
# 7. Main pipeline
# =========================================================

def main(cfg: Config):
    if Mamba is None:
        raise ImportError("mamba_ssm is not installed. Please run: pip install mamba-ssm")

    set_seed(cfg.seed)
    ensure_dir(cfg.output_dir)

    loader = NGAFIDRawLoader(
        cfg.data_dir,
        index_col=cfg.index_col,
        force_seq_channels=cfg.force_seq_channels
    )
    pack = loader.load(target_col=cfg.target_col)

    ids = pack["ids"]
    seqs = pack["seqs"]
    header_df = pack["header_df"]
    labels = pack["labels"]
    lengths = pack["lengths"]
    mins = pack["mins"]
    maxs = pack["maxs"]
    bad_ids = pack["bad_ids"]

    assert len(ids) == len(seqs) == len(labels) == len(header_df), \
        "ids / seqs / labels / header_df length mismatch"
    assert list(header_df.index) == list(ids), \
        "header_df.index and ids are not aligned"

    dims = [x.shape[1] for x in seqs]
    assert len(set(dims)) == 1, f"Inconsistent input dims: {set(dims)}"

    static_cols = (
        infer_static_columns(header_df, cfg.target_col, cfg.fold_col, cfg.extra_drop_cols)
        if cfg.use_static_features else []
    )
    static_df = header_df[static_cols].copy() if len(static_cols) > 0 else pd.DataFrame(index=header_df.index)

    seq_in_dim = cfg.seq_in_dim_override or seqs[0].shape[1]
    device = "cuda" if torch.cuda.is_available() else "cpu"

    target_dist = pd.Series(labels).value_counts().sort_index()

    info = {
        "num_samples": int(len(ids)),
        "num_bad_ids": int(len(bad_ids)),
        "sequence_channels": int(seqs[0].shape[1]),
        "length_min": int(lengths.min()),
        "length_median": float(np.median(lengths)),
        "length_max": int(lengths.max()),
        "static_cols": list(static_cols),
        "target_distribution": {str(int(k)): int(v) for k, v in target_dist.items()},
    }
    with open(os.path.join(cfg.output_dir, "dataset_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)

    print("=" * 90)
    print("Dataset loaded")
    print(f"num_samples: {len(ids)}")
    print(f"num_bad_ids: {len(bad_ids)}")
    print(f"seq_in_dim: {seq_in_dim}")
    print(f"label_dist: {dict(target_dist)}")
    print(f"static_cols ({len(static_cols)}): {static_cols}")

    fold_ids = get_fold_splits(header_df, cfg.fold_col)
    fold_results = []

    id_to_pos = {id_: i for i, id_ in enumerate(ids)}

    for fold in fold_ids:
        print("=" * 90)
        print(f"Fold {fold}")

        train_fold_ids = header_df.index[header_df[cfg.fold_col].astype(int) != fold].tolist()
        val_fold_ids = header_df.index[header_df[cfg.fold_col].astype(int) == fold].tolist()

        train_pos = [id_to_pos[i] for i in train_fold_ids]
        val_pos = [id_to_pos[i] for i in val_fold_ids]

        train_ids = [ids[i] for i in train_pos]
        val_ids = [ids[i] for i in val_pos]

        train_seqs = [seqs[i] for i in train_pos]
        val_seqs = [seqs[i] for i in val_pos]

        y_train = labels[train_pos]
        y_val = labels[val_pos]

        train_static_df = static_df.loc[train_fold_ids].copy()
        val_static_df = static_df.loc[val_fold_ids].copy()

        overlap = set(train_ids) & set(val_ids)
        assert len(overlap) == 0, f"train/val overlap detected: {len(overlap)}"

        print(f"train size: {len(train_ids)}")
        print(f"val size: {len(val_ids)}")
        print(f"train label dist: {np.bincount(y_train)}")
        print(f"val label dist: {np.bincount(y_val)}")
        print(f"train/val overlap: {len(overlap)}")

        pre = FoldPreprocessor(
            mins=mins,
            maxs=maxs,
            use_stats_minmax=cfg.use_stats_minmax,
            eps=cfg.norm_eps
        )
        pre.fit(train_seqs, train_static_df)

        train_seqs = [pre.transform_sequence(x) for x in train_seqs]
        val_seqs = [pre.transform_sequence(x) for x in val_seqs]

        x_static_train = pre.transform_static(train_static_df)
        x_static_val = pre.transform_static(val_static_df)

        train_ds = NGAFIDSequenceDataset(train_ids, train_seqs, x_static_train, y_train)
        val_ds = NGAFIDSequenceDataset(val_ids, val_seqs, x_static_val, y_val)

        train_sampler = LengthBucketBatchSampler(
            lengths=train_ds.lengths,
            batch_size=cfg.batch_size,
            boundaries=cfg.bucket_boundaries,
            shuffle=True,
            drop_last=False,
        )
        val_sampler = LengthBucketBatchSampler(
            lengths=val_ds.lengths,
            batch_size=cfg.batch_size,
            boundaries=cfg.bucket_boundaries,
            shuffle=False,
            drop_last=False,
        )

        train_loader = DataLoader(
            train_ds,
            batch_sampler=train_sampler,
            num_workers=cfg.num_workers,
            pin_memory=(device == "cuda"),
            collate_fn=collate_fn,
        )
        val_loader = DataLoader(
            val_ds,
            batch_sampler=val_sampler,
            num_workers=cfg.num_workers,
            pin_memory=(device == "cuda"),
            collate_fn=collate_fn,
        )

        model = CNNBiMambaClassifier(
            seq_in_dim=seq_in_dim,
            static_dim=x_static_train.shape[1],
            cnn_dim=cfg.cnn_dim,
            stem_kernel=cfg.stem_kernel,
            stem_stride=cfg.stem_stride,
            mamba_layers=cfg.mamba_layers,
            mamba_d_state=cfg.mamba_d_state,
            mamba_d_conv=cfg.mamba_d_conv,
            mamba_expand=cfg.mamba_expand,
            static_hidden=cfg.static_hidden,
            static_out=cfg.static_out,
            num_classes=cfg.num_classes,
            dropout=cfg.dropout,
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        criterion = nn.CrossEntropyLoss()
        scaler = torch.cuda.amp.GradScaler(enabled=(cfg.amp and device == "cuda"))
        stopper = EarlyStopping(patience=cfg.patience, mode="max")

        history_rows = []
        best_epoch = -1
        best_acc = -1.0
        best_model_path = os.path.join(cfg.output_dir, f"best_fold_{fold}.pt")

        for epoch in range(1, cfg.epochs + 1):
            train_metrics = run_epoch(
                model, train_loader, optimizer, criterion, device,
                scaler=scaler, train=True, amp=cfg.amp
            )
            val_metrics = run_epoch(
                model, val_loader, optimizer, criterion, device,
                scaler=scaler, train=False, amp=cfg.amp
            )

            row = {
                "fold": fold,
                "epoch": epoch,
                "train_loss": train_metrics["loss"],
                "train_acc": train_metrics["acc"],
                "train_precision": train_metrics["precision"],
                "train_recall": train_metrics["recall"],
                "train_f1": train_metrics["f1"],
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["acc"],
                "val_precision": val_metrics["precision"],
                "val_recall": val_metrics["recall"],
                "val_f1": val_metrics["f1"],
            }
            history_rows.append(row)

            print(
                f"[Fold {fold}][Epoch {epoch:02d}] "
                f"train_loss={row['train_loss']:.6f} train_acc={row['train_acc']:.6f} | "
                f"val_loss={row['val_loss']:.6f} val_acc={row['val_acc']:.6f} val_f1={row['val_f1']:.6f}"
            )

            improved = stopper.step(row["val_acc"])
            if improved:
                best_epoch = epoch
                best_acc = row["val_acc"]
                torch.save(model.state_dict(), best_model_path)

            if stopper.stop:
                print(f"Early stopping on fold {fold} at epoch {epoch}")
                break

        hist_df = pd.DataFrame(history_rows)
        hist_df.to_csv(os.path.join(cfg.output_dir, f"history_fold_{fold}.csv"), index=False)

        model.load_state_dict(torch.load(best_model_path, map_location=device))
        final_val = run_epoch(
            model, val_loader, optimizer, criterion, device,
            scaler=scaler, train=False, amp=cfg.amp
        )

        pd.DataFrame(final_val["pred_rows"]).to_csv(
            os.path.join(cfg.output_dir, f"predictions_fold_{fold}.csv"),
            index=False
        )

        fold_results.append({
            "fold": fold,
            "best_epoch": best_epoch,
            "best_val_acc_during_train": best_acc,
            "val_acc": final_val["acc"],
            "val_precision": final_val["precision"],
            "val_recall": final_val["recall"],
            "val_f1": final_val["f1"],
        })

    result_df = pd.DataFrame(fold_results).sort_values("fold")
    result_df.to_csv(os.path.join(cfg.output_dir, "fold_results.csv"), index=False)

    mean_acc = result_df["val_acc"].mean()
    std_acc = result_df["val_acc"].std(ddof=1)

    summary = {
        "fold_results": fold_results,
        "mean_acc": float(mean_acc),
        "std_acc": float(std_acc),
        "config": asdict(cfg),
    }
    with open(os.path.join(cfg.output_dir, "summary.json"), "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("=" * 90)
    for _, r in result_df.iterrows():
        print(
            f"Fold {int(r['fold'])}: "
            f"best_epoch={int(r['best_epoch'])}, "
            f"best_val_acc_during_train={r['best_val_acc_during_train']:.6f}, "
            f"final_val_acc={r['val_acc']:.6f}"
        )
    print(f"Mean ± Std: {mean_acc:.6f} ± {std_acc:.6f}")


if __name__ == "__main__":
    main(CFG)

Dataset loaded
num_samples: 11446
num_bad_ids: 0
seq_in_dim: 23
label_dist: {0: 5844, 1: 5602}
static_cols (0): []
Fold 0
train size: 9156
val size: 2290
train label dist: [4677 4479]
val label dist: [1167 1123]
train/val overlap: 0
[Fold 0][Epoch 01] train_loss=0.691919 train_acc=0.548274 | val_loss=0.730249 val_acc=0.500437 val_f1=0.639572
[Fold 0][Epoch 02] train_loss=0.647342 train_acc=0.620031 | val_loss=0.710807 val_acc=0.575546 val_f1=0.673167
[Fold 0][Epoch 03] train_loss=0.610498 train_acc=0.662407 | val_loss=0.647906 val_acc=0.603930 val_f1=0.493579
[Fold 0][Epoch 04] train_loss=0.576257 train_acc=0.700306 | val_loss=0.701744 val_acc=0.610044 val_f1=0.456482
[Fold 0][Epoch 05] train_loss=0.562040 train_acc=0.715815 | val_loss=0.530498 val_acc=0.744105 val_f1=0.740478
[Fold 0][Epoch 06] train_loss=0.548707 train_acc=0.724661 | val_loss=0.599548 val_acc=0.688646 val_f1=0.726296
[Fold 0][Epoch 07] train_loss=0.535656 train_acc=0.736020 | val_loss=0.557278 val_acc=0.704367 val_f1